In [ ]:
# Cell 1: Import libraries and load both datasets
from datasets import load_dataset
import pandas as pd

# Load sentiment dataset (Twitter sentiment: positive/negative/neutral)
sentiment_data = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Load intent dataset (banking customer queries: 77 intents)
# Using the mteb mirror since it's already in Parquet format
intent_data = load_dataset("mteb/banking77")

print("Sentiment dataset loaded:", sentiment_data)
print("\nIntent dataset loaded:", intent_data)

Sentiment dataset loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Intent dataset loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})


In [3]:
# Cell 2: Look at the actual data
sentiment_df = sentiment_data['train'].to_pandas()
intent_df = intent_data['train'].to_pandas()

print("SENTIMENT DATA SAMPLE:")
print(sentiment_df.head())
print("\nSentiment label distribution:")
print(sentiment_df['label'].value_counts())

print("\n\nINTENT DATA SAMPLE:")
print(intent_df.head())
print("\nNumber of unique intents:", intent_df['label'].nunique())
print("\nSample intent names:")
print(intent_df['label_text'].unique()[:10])

SENTIMENT DATA SAMPLE:
                                                text  label
0  "QT @user In the original draft of the 7th boo...      2
1  "Ben Smith / Smith (concussion) remains out of...      1
2  Sorry bout the stream last night I crashed out...      1
3  Chase Headley's RBI double in the 8th inning o...      1
4  @user Alciato: Bee will invest 150 million in ...      2

Sentiment label distribution:
label
1    20673
2    17849
0     7093
Name: count, dtype: int64


INTENT DATA SAMPLE:
                                                text  label    label_text
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival
3  Can I track my card while it is in the process...     11  card_arrival
4  How do I know if I will get my card, or if it ...     11  card_arrival

Number of unique intents: 77

Sample intent names:
<Ar

In [4]:
# Cell 3: One-time downloads for text processing
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# Cell 4: Text cleaning function
import re
import spacy
from nltk.corpus import stopwords

# Load spaCy model once (reused for every row — loading it repeatedly is slow)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])  # disable unused parts for speed
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)         # remove URLs
    text = re.sub(r'@\w+', '', text)                   # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)               # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()           # remove extra whitespace

    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text not in stop_words and len(token.text) > 1]
    return ' '.join(tokens)

# Quick test on one example
sample = sentiment_df['text'].iloc[0]
print("BEFORE:", sample)
print("AFTER:", clean_text(sample))

BEFORE: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
AFTER: qt original draft th book remus lupin survive battle hogwart happybirthdayremuslupin


In [6]:
# Cell 5: Apply cleaning to full sentiment dataset (this will take a few minutes)
from tqdm import tqdm
tqdm.pandas()

sentiment_df['clean_text'] = sentiment_df['text'].progress_apply(clean_text)

print(sentiment_df[['text', 'clean_text']].head())

100%|██████████| 45615/45615 [06:45<00:00, 112.48it/s]


                                                text  \
0  "QT @user In the original draft of the 7th boo...   
1  "Ben Smith / Smith (concussion) remains out of...   
2  Sorry bout the stream last night I crashed out...   
3  Chase Headley's RBI double in the 8th inning o...   
4  @user Alciato: Bee will invest 150 million in ...   

                                          clean_text  
0  qt original draft th book remus lupin survive ...  
1  ben smith smith concussion remain lineup thurs...  
2  sorry bout stream last night crash tonight sur...  
3  chase headley rbi double th inning david price...  
4  alciato bee invest million january another sum...  


In [7]:
# Cell 6: Apply cleaning to intent dataset
intent_df['clean_text'] = intent_df['text'].progress_apply(clean_text)
print(intent_df[['text', 'clean_text']].head())

  0%|          | 15/9993 [00:00<02:10, 76.51it/s]

100%|██████████| 9993/9993 [01:22<00:00, 121.35it/s]


                                                text  \
0                     I am still waiting on my card?   
1  What can I do if my card still hasn't arrived ...   
2  I have been waiting over a week. Is the card s...   
3  Can I track my card while it is in the process...   
4  How do I know if I will get my card, or if it ...   

                    clean_text  
0              still wait card  
1   card still not arrive week  
2    wait week card still come  
3  track card process delivery  
4           know get card lose  


In [8]:
# Cell 7: Save cleaned datasets to the data/ folder
sentiment_df.to_csv('../data/sentiment_cleaned.csv', index=False)
intent_df.to_csv('../data/intent_cleaned.csv', index=False)

print("Saved successfully.")

Saved successfully.


In [9]:
# Cell 8: Train/test split + TF-IDF + Logistic Regression baseline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Split cleaned data
X = sentiment_df['clean_text']
y = sentiment_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train baseline model (class_weight='balanced' helps with your imbalanced classes)
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

              precision    recall  f1-score   support

    negative       0.44      0.64      0.52      1419
     neutral       0.67      0.59      0.63      4134
    positive       0.69      0.65      0.67      3570

    accuracy                           0.62      9123
   macro avg       0.60      0.63      0.61      9123
weighted avg       0.64      0.62      0.63      9123



In [10]:
# Cell 9: Save the sentiment model and vectorizer
import joblib

joblib.dump(model, '../models/sentiment_model.pkl')
joblib.dump(vectorizer, '../models/sentiment_vectorizer.pkl')

print("Sentiment model saved.")

Sentiment model saved.


In [11]:
# Cell 10: Train/test split + TF-IDF + Logistic Regression for intent
X_intent = intent_df['clean_text']
y_intent = intent_df['label']

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_intent, y_intent, test_size=0.2, random_state=42, stratify=y_intent
)

vectorizer_intent = TfidfVectorizer(max_features=5000)
X_train_i_tfidf = vectorizer_intent.fit_transform(X_train_i)
X_test_i_tfidf = vectorizer_intent.transform(X_test_i)

model_intent = LogisticRegression(max_iter=1000, class_weight='balanced')
model_intent.fit(X_train_i_tfidf, y_train_i)

y_pred_i = model_intent.predict(X_test_i_tfidf)
print(classification_report(y_test_i, y_pred_i))

              precision    recall  f1-score   support

           0       1.00      0.88      0.93        32
           1       1.00      0.95      0.98        22
           2       0.93      1.00      0.96        25
           3       0.89      1.00      0.94        17
           4       1.00      0.88      0.94        25
           5       0.81      0.76      0.79        34
           6       0.90      0.97      0.93        36
           7       0.89      0.81      0.85        31
           8       0.88      0.97      0.92        31
           9       1.00      0.92      0.96        26
          10       0.91      0.83      0.87        12
          11       0.81      0.94      0.87        31
          12       0.86      0.82      0.84        22
          13       0.96      0.93      0.95        28
          14       0.62      0.73      0.67        22
          15       0.84      0.84      0.84        38
          16       0.77      0.88      0.82        34
          17       0.96    

In [12]:
# Cell 11: Save the intent model and vectorizer
joblib.dump(model_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_intent, '../models/intent_vectorizer.pkl')

print("Intent model saved.")

Intent model saved.


In [13]:
# Cell 12: Sanity test both models on made-up examples
test_sentences = [
    "This product is absolutely amazing, I love it!",
    "Terrible service, I want a refund immediately.",
    "It's okay, nothing special but not bad either.",
    "My card hasn't arrived yet, it's been two weeks.",
    "Can you tell me your working hours?"
]

for text in test_sentences:
    cleaned = clean_text(text)

    sent_vec = vectorizer.transform([cleaned])
    sentiment_pred = model.predict(sent_vec)[0]
    sentiment_label = ['negative', 'neutral', 'positive'][sentiment_pred]

    intent_vec = vectorizer_intent.transform([cleaned])
    intent_pred = model_intent.predict(intent_vec)[0]
    intent_label = intent_df[intent_df['label'] == intent_pred]['label_text'].iloc[0]

    print(f"Text: {text}")
    print(f"  → Sentiment: {sentiment_label} | Intent: {intent_label}\n")

Text: This product is absolutely amazing, I love it!
  → Sentiment: positive | Intent: request_refund

Text: Terrible service, I want a refund immediately.
  → Sentiment: negative | Intent: request_refund

Text: It's okay, nothing special but not bad either.
  → Sentiment: negative | Intent: receiving_money

Text: My card hasn't arrived yet, it's been two weeks.
  → Sentiment: negative | Intent: card_arrival

Text: Can you tell me your working hours?
  → Sentiment: neutral | Intent: pending_top_up



In [14]:
# Cell 13: Save intent label map so pipeline.py can use it independently
import json

intent_label_map = dict(zip(intent_df['label'], intent_df['label_text']))
intent_label_map = {int(k): v for k, v in intent_label_map.items()}  # ensure keys are plain ints

with open('../models/intent_label_map.json', 'w') as f:
    json.dump(intent_label_map, f)

print("Saved. Total intents:", len(intent_label_map))

Saved. Total intents: 77
